In [0]:
dest_ai_path='development_042_silver_sandbox.demand_forecast.'
like_games_path_old='development_042_silver_sandbox.demand_forecast.clustering_core3_like_games'
new_like_games_path=dest_ai_path+"similar_games_ai"
new_path=dest_ai_path+"pred_games_ai"
old_path='development_042_silver_sandbox.demand_forecast.time_series_extrapolated'

In [0]:
old_like_games_df = spark.table(
    "development_042_silver_sandbox.demand_forecast.clustering_core3_like_games"
)

similar_games_ai_df = spark.table(
    "development_042_silver_sandbox.demand_forecast.similar_games_ai"
)

pred_games_ai_df = spark.table(
    "development_042_silver_sandbox.demand_forecast.pred_games_ai"
)

time_series_extrapolated_df = spark.table(
    "development_042_silver_sandbox.demand_forecast.time_series_extrapolated"
)

In [0]:
display(old_like_games_df)
display(similar_games_ai_df)
display(pred_games_ai_df)
display(time_series_extrapolated_df)

In [0]:
%pip install plotly

In [0]:
import re
import unicodedata

import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pyspark.sql import functions as F


# Old-analysis filters
OLD_EXPERIMENT = "Ricky's weights( high theme and feature)"
OLD_BEST_MODEL = "Polynomial_deg2"
MIN_ACTUAL_MONTHS = 4


def normalize_game_name(value):
    """Normalize punctuation and spacing so differently formatted names match."""
    if value is None:
        return ""

    value = unicodedata.normalize("NFKD", str(value))
    value = value.encode("ascii", "ignore").decode("ascii")
    value = value.lower().replace("&", " and ")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return " ".join(value.split())


# Old like games: keep only Ricky's experiment.
old_likes_pd = (
    old_like_games_df
    .filter(F.col("experiment_name") == OLD_EXPERIMENT)
    .select(
        "target_game_name",
        "dataset_split",
        "neighbor_rank",
        "game_name",
        "cosine_distance",
        "train__theme_name",
        "train__form_factor",
        "train__mechanic_segment",
    )
    .toPandas()
)

# AI like games: includes cosine and Euclidean results.
ai_likes_pd = (
    similar_games_ai_df
    .select(
        "target_game_name",
        "target_display_name",
        "analysis_split",
        "game_name",
        "neighbor_display_name",
        "neighbor_rank",
        "distance",
        "distance_metric",
    )
    .toPandas()
)

# Old forecasts: keep Ricky's experiment and the selected best model.
old_results_pd = (
    time_series_extrapolated_df
    .filter(
        (F.col("experiment_name") == OLD_EXPERIMENT)
        & (F.col("best_model") == OLD_BEST_MODEL)
    )
    .select(
        "target_game_name",
        "dataset_split",
        "release_month",
        "ForecastUnitBaseline",
        "shifted_actual_expr1_sum",
    )
    .toPandas()
)

# AI-based predictions and observed actual monthly sales.
ai_predictions_pd = (
    pred_games_ai_df
    .select(
        "target_game_name",
        "target_display_name",
        "analysis_split",
        "month_index",
        "distance_metric",
        "sales_mean",
        "actual_sales_cumsum",
    )
    .toPandas()
)


# Normalize game names so punctuation and "&"/"and" differences do not block matches.
for df in [old_likes_pd, ai_likes_pd, old_results_pd, ai_predictions_pd]:
    df["_game_key"] = df["target_game_name"].map(normalize_game_name)


# Count distinct months with actual sales, combining valid and test.
old_actual_month_counts = (
    old_results_pd
    .dropna(subset=["shifted_actual_expr1_sum"])
    .drop_duplicates(["_game_key", "release_month"])
    .groupby("_game_key")
    .size()
)

ai_actual_month_counts = (
    ai_predictions_pd
    .dropna(subset=["actual_sales_cumsum"])
    # Actuals repeat across cosine and Euclidean rows, so count each month once.
    .drop_duplicates(["_game_key", "month_index"])
    .groupby("_game_key")
    .size()
)

eligible_game_keys = set(
    old_actual_month_counts[
        old_actual_month_counts >= MIN_ACTUAL_MONTHS
    ].index
) & set(
    ai_actual_month_counts[
        ai_actual_month_counts >= MIN_ACTUAL_MONTHS
    ].index
)

# Require the game to exist in all four objects.
eligible_game_keys &= set(old_likes_pd["_game_key"])
eligible_game_keys &= set(ai_likes_pd["_game_key"])
eligible_game_keys &= set(old_results_pd["_game_key"])
eligible_game_keys &= set(ai_predictions_pd["_game_key"])

if not eligible_game_keys:
    raise ValueError(
        "No matching games have at least "
        f"{MIN_ACTUAL_MONTHS} months of actual sales in both analyses."
    )


# Use the AI display name for the dropdown when available.
display_names = (
    ai_predictions_pd
    .dropna(subset=["target_display_name"])
    .drop_duplicates("_game_key")
    .set_index("_game_key")["target_display_name"]
    .to_dict()
)

game_options = [
    (display_names.get(game_key, game_key.title()), game_key)
    for game_key in sorted(eligible_game_keys)
]

game_dropdown = widgets.Dropdown(
    options=game_options,
    description="Game:",
    layout=widgets.Layout(width="600px"),
    style={"description_width": "initial"},
)

output = widgets.Output()


def show_game(game_key):
    with output:
        clear_output(wait=True)

        # Old like games, combining valid and test.
        old_like_rows = (
            old_likes_pd[old_likes_pd["_game_key"] == game_key]
            .sort_values(["dataset_split", "neighbor_rank"])
            .rename(
                columns={
                    "dataset_split": "Split",
                    "game_name": "Like game",
                    "neighbor_rank": "Rank",
                    "cosine_distance": "Cosine distance",
                    "train__theme_name": "Theme",
                    "train__form_factor": "Form factor",
                    "train__mechanic_segment": "Mechanic segment",
                }
            )
        )

        # AI cosine like games, combining valid and test.
        cosine_rows = (
            ai_likes_pd[
                (ai_likes_pd["_game_key"] == game_key)
                & (ai_likes_pd["distance_metric"].astype(str).str.lower() == "cosine")
            ]
            .sort_values(["analysis_split", "neighbor_rank"])
            .rename(
                columns={
                    "analysis_split": "Split",
                    "neighbor_display_name": "Like game",
                    "neighbor_rank": "Rank",
                    "distance": "Cosine distance",
                }
            )
        )

        # AI Euclidean like games, combining valid and test.
        euclidean_rows = (
            ai_likes_pd[
                (ai_likes_pd["_game_key"] == game_key)
                & (ai_likes_pd["distance_metric"].astype(str).str.lower() == "euclidean")
            ]
            .sort_values(["analysis_split", "neighbor_rank"])
            .rename(
                columns={
                    "analysis_split": "Split",
                    "neighbor_display_name": "Like game",
                    "neighbor_rank": "Rank",
                    "distance": "Euclidean distance",
                }
            )
        )

        old_display = old_like_rows[
            [
                "Split",
                "Rank",
                "Like game",
                "Cosine distance",
                "Theme",
                "Form factor",
                "Mechanic segment",
            ]
        ]

        cosine_display = cosine_rows[
            ["Split", "Rank", "Like game", "Cosine distance"]
        ]

        euclidean_display = euclidean_rows[
            ["Split", "Rank", "Like game", "Euclidean distance"]
        ]

        cards = [
            ("Old like games — Ricky's weights", old_display),
            ("AI like games — cosine", cosine_display),
            ("AI like games — Euclidean", euclidean_display),
        ]

        card_html = ""
        for title, table_df in cards:
            card_html += (
                '<div style="flex:1; min-width:300px; padding:8px;">'
                f"<h4>{title}</h4>"
                f"{table_df.to_html(index=False, na_rep='', border=0)}"
                "</div>"
            )

        display(
            HTML(
                '<div style="display:flex; gap:12px; align-items:flex-start; '
                'overflow-x:auto;">'
                + card_html
                + "</div>"
            )
        )

        # Old ForecastUnitBaseline curve, combining valid and test.
        old_curve = old_results_pd[
            old_results_pd["_game_key"] == game_key
        ].copy()

        old_curve["release_month"] = pd.to_numeric(
            old_curve["release_month"], errors="coerce"
        )
        old_curve["ForecastUnitBaseline"] = pd.to_numeric(
            old_curve["ForecastUnitBaseline"], errors="coerce"
        )

        old_curve = (
            old_curve
            .dropna(subset=["release_month", "ForecastUnitBaseline"])
            .groupby("release_month", as_index=False)
            .agg(ForecastUnitBaseline=("ForecastUnitBaseline", "mean"))
            .sort_values("release_month")
        )

        # AI prediction results, combining valid and test.
        ai_curve = ai_predictions_pd[
            ai_predictions_pd["_game_key"] == game_key
        ].copy()

        ai_curve["month_number"] = (
            pd.to_numeric(ai_curve["month_index"], errors="coerce") + 1
        )
        ai_curve["sales_mean"] = pd.to_numeric(
            ai_curve["sales_mean"], errors="coerce"
        )
        ai_curve["actual_sales_cumsum"] = pd.to_numeric(
            ai_curve["actual_sales_cumsum"], errors="coerce"
        )

        # Actual sales repeat across distance metrics; keep one actual per month.
        actual_curve = (
            ai_curve
            .dropna(subset=["actual_sales_cumsum"])
            .sort_values("month_number")
            .drop_duplicates("month_number")
        )

        fig = go.Figure()

        fig.add_trace(
            go.Scatter(
                x=old_curve["release_month"],
                y=old_curve["ForecastUnitBaseline"],
                mode="lines+markers",
                name="Old ForecastUnitBaseline",
            )
        )

        fig.add_trace(
            go.Scatter(
                x=actual_curve["month_number"],
                y=actual_curve["actual_sales_cumsum"],
                mode="lines+markers",
                name="Actual sales",
            )
        )

        for metric, label in [
            ("cosine", "Predicted sales mean — cosine"),
            ("euclidean", "Predicted sales mean — Euclidean"),
        ]:
            metric_curve = (
                ai_curve[
                    ai_curve["distance_metric"].astype(str).str.lower() == metric
                ]
                .dropna(subset=["month_number", "sales_mean"])
                .groupby("month_number", as_index=False)
                .agg(sales_mean=("sales_mean", "mean"))
                .sort_values("month_number")
            )

            fig.add_trace(
                go.Scatter(
                    x=metric_curve["month_number"],
                    y=metric_curve["sales_mean"],
                    mode="lines+markers",
                    name=label,
                )
            )

        game_label = display_names.get(game_key, game_key.title())

        fig.update_layout(
            title=f"{game_label} — valid and test combined",
            xaxis_title="Month since release",
            yaxis_title="Sales",
            hovermode="x unified",
            legend_title="Series",
        )

        # Embed Plotly's JavaScript so Databricks can render the chart.
        displayHTML(
            fig.to_html(full_html=False, include_plotlyjs=True)
        )


def on_game_change(change):
    if change["name"] == "value":
        show_game(change["new"])


game_dropdown.observe(on_game_change, names="value")

display(game_dropdown, output)
show_game(game_dropdown.value)

In [0]:
import re
import unicodedata

import pandas as pd
from pyspark.sql import functions as F


OLD_EXPERIMENT = "Ricky's weights( high theme and feature)"
OLD_BEST_MODEL = "Polynomial_deg2"
MIN_ACTUAL_MONTHS = 4


def normalize_game_name(value):
    if value is None:
        return ""

    value = unicodedata.normalize("NFKD", str(value))
    value = value.encode("ascii", "ignore").decode("ascii")
    value = value.lower().replace("&", " and ")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return " ".join(value.split())


# Keep only targets represented in both like-game analyses.
perf_old_like_names = (
    old_like_games_df
    .filter(F.col("experiment_name") == OLD_EXPERIMENT)
    .select("target_game_name")
    .distinct()
    .toPandas()
)

perf_ai_like_names = (
    similar_games_ai_df
    .select("target_game_name")
    .distinct()
    .toPandas()
)

perf_old_like_names["_game_key"] = (
    perf_old_like_names["target_game_name"].map(normalize_game_name)
)
perf_ai_like_names["_game_key"] = (
    perf_ai_like_names["target_game_name"].map(normalize_game_name)
)

perf_shared_like_keys = (
    set(perf_old_like_names["_game_key"])
    & set(perf_ai_like_names["_game_key"])
)


# Old forecast and cumulative actuals.
perf_old_pd = (
    time_series_extrapolated_df
    .filter(
        (F.col("experiment_name") == OLD_EXPERIMENT)
        & (F.col("best_model") == OLD_BEST_MODEL)
    )
    .select(
        "target_game_name",
        "release_month",
        "ForecastUnitBaseline",
        "shifted_actual_cumsum_expr1_sum",
    )
    .toPandas()
)

# New cumulative predictions and cumulative actuals.
perf_new_pd = (
    pred_games_ai_df
    .select(
        "target_game_name",
        "target_display_name",
        "month_index",
        "distance_metric",
        "sales_mean",
        "actual_sales_cumsum",
    )
    .toPandas()
)

perf_old_pd["_game_key"] = perf_old_pd["target_game_name"].map(
    normalize_game_name
)
perf_new_pd["_game_key"] = perf_new_pd["target_game_name"].map(
    normalize_game_name
)

perf_old_pd["month_number"] = pd.to_numeric(
    perf_old_pd["release_month"], errors="coerce"
)
perf_old_pd["ForecastUnitBaseline"] = pd.to_numeric(
    perf_old_pd["ForecastUnitBaseline"], errors="coerce"
)
perf_old_pd["shifted_actual_cumsum_expr1_sum"] = pd.to_numeric(
    perf_old_pd["shifted_actual_cumsum_expr1_sum"], errors="coerce"
)

perf_new_pd["month_number"] = (
    pd.to_numeric(perf_new_pd["month_index"], errors="coerce") + 1
)
perf_new_pd["sales_mean"] = pd.to_numeric(
    perf_new_pd["sales_mean"], errors="coerce"
)
perf_new_pd["actual_sales_cumsum"] = pd.to_numeric(
    perf_new_pd["actual_sales_cumsum"], errors="coerce"
)
perf_new_pd["distance_metric"] = (
    perf_new_pd["distance_metric"].astype(str).str.lower().str.strip()
)


# Combine valid/test and collapse duplicate game-month rows.
perf_old_monthly = (
    perf_old_pd
    .dropna(subset=["_game_key", "month_number"])
    .groupby(["_game_key", "month_number"], as_index=False)
    .agg(
        old_prediction=("ForecastUnitBaseline", "mean"),
        old_actual_cumsum=("shifted_actual_cumsum_expr1_sum", "first"),
    )
)

perf_new_monthly = (
    perf_new_pd
    .dropna(subset=["_game_key", "month_number"])
    .groupby(
        ["_game_key", "month_number", "distance_metric"],
        as_index=False,
    )
    .agg(
        prediction=("sales_mean", "mean"),
        actual_sales_cumsum=("actual_sales_cumsum", "first"),
    )
)


# Require at least four months of cumulative actual sales in both results.
perf_old_actual_counts = (
    perf_old_monthly
    .dropna(subset=["old_actual_cumsum"])
    .groupby("_game_key")["month_number"]
    .nunique()
)

perf_new_actual_counts = (
    perf_new_monthly
    .dropna(subset=["actual_sales_cumsum"])
    .groupby("_game_key")["month_number"]
    .nunique()
)

perf_eligible_keys = (
    perf_shared_like_keys
    & set(
        perf_old_actual_counts[
            perf_old_actual_counts >= MIN_ACTUAL_MONTHS
        ].index
    )
    & set(
        perf_new_actual_counts[
            perf_new_actual_counts >= MIN_ACTUAL_MONTHS
        ].index
    )
)

if not perf_eligible_keys:
    raise ValueError(
        "No shared games have at least "
        f"{MIN_ACTUAL_MONTHS} months of cumulative actual sales."
    )


# Use one common cumulative actual series for all three models.
perf_actuals = (
    perf_new_monthly
    .dropna(subset=["actual_sales_cumsum"])
    .groupby(["_game_key", "month_number"], as_index=False)
    .agg(actual=("actual_sales_cumsum", "first"))
)

perf_ai_predictions_wide = (
    perf_new_monthly
    .pivot_table(
        index=["_game_key", "month_number"],
        columns="distance_metric",
        values="prediction",
        aggfunc="mean",
    )
    .reset_index()
)
perf_ai_predictions_wide.columns.name = None

if not {"cosine", "euclidean"}.issubset(perf_ai_predictions_wide.columns):
    raise ValueError(
        "The new prediction data must contain both cosine and Euclidean rows."
    )


# Keep only game-months with actuals and predictions from all three models.
perf_comparison = (
    perf_actuals
    .merge(
        perf_ai_predictions_wide,
        on=["_game_key", "month_number"],
        how="inner",
    )
    .merge(
        perf_old_monthly[["_game_key", "month_number", "old_prediction"]],
        on=["_game_key", "month_number"],
        how="inner",
    )
)

perf_comparison = (
    perf_comparison[
        perf_comparison["_game_key"].isin(perf_eligible_keys)
    ]
    .dropna(subset=["actual", "old_prediction", "cosine", "euclidean"])
)

# Keep games with at least four fully comparable months.
perf_common_month_counts = perf_comparison.groupby("_game_key").size()
perf_final_keys = set(
    perf_common_month_counts[
        perf_common_month_counts >= MIN_ACTUAL_MONTHS
    ].index
)
perf_comparison = perf_comparison[
    perf_comparison["_game_key"].isin(perf_final_keys)
].copy()

if perf_comparison.empty:
    raise ValueError(
        "No games have at least four months with cumulative actuals and "
        "predictions available for all three models."
    )


perf_name_lookup = (
    perf_new_pd
    .dropna(subset=["target_display_name"])
    .drop_duplicates("_game_key")
    .set_index("_game_key")["target_display_name"]
    .to_dict()
)
perf_comparison["game_name"] = perf_comparison["_game_key"].map(
    lambda key: perf_name_lookup.get(key, key)
)


# Same output structure expected by Cell 2.
perf_evaluation_df = pd.concat(
    [
        perf_comparison[
            ["_game_key", "game_name", "month_number", "actual", "old_prediction"]
        ]
        .rename(columns={"old_prediction": "prediction"})
        .assign(model="Old — ForecastUnitBaseline"),

        perf_comparison[
            ["_game_key", "game_name", "month_number", "actual", "cosine"]
        ]
        .rename(columns={"cosine": "prediction"})
        .assign(model="New — cosine"),

        perf_comparison[
            ["_game_key", "game_name", "month_number", "actual", "euclidean"]
        ]
        .rename(columns={"euclidean": "prediction"})
        .assign(model="New — Euclidean"),
    ],
    ignore_index=True,
)

print(f"Games included: {perf_comparison['_game_key'].nunique()}")
print(f"Comparable game-months per model: {len(perf_comparison)}")

In [0]:
import numpy as np
import pandas as pd


def calculate_metrics(df, group_columns):
    rows = []

    grouped = (
        df.groupby(group_columns, dropna=False)
        if group_columns
        else [((), df)]
    )

    for group_key, group in grouped:
        actual = group["actual"].astype(float).to_numpy()
        prediction = group["prediction"].astype(float).to_numpy()

        error = prediction - actual
        absolute_error = np.abs(error)

        ss_total = np.sum((actual - np.mean(actual)) ** 2)
        r2 = (
            1 - np.sum(error ** 2) / ss_total
            if len(actual) >= 2 and ss_total > 0
            else np.nan
        )

        nonzero_actual = actual != 0
        mape = (
            np.mean(
                absolute_error[nonzero_actual]
                / np.abs(actual[nonzero_actual])
            ) * 100
            if nonzero_actual.any()
            else np.nan
        )

        wape_denominator = np.sum(np.abs(actual))
        wape = (
            np.sum(absolute_error) / wape_denominator * 100
            if wape_denominator > 0
            else np.nan
        )

        smape_denominator = np.abs(actual) + np.abs(prediction)
        valid_smape = smape_denominator != 0
        smape = (
            np.mean(
                2 * absolute_error[valid_smape]
                / smape_denominator[valid_smape]
            ) * 100
            if valid_smape.any()
            else np.nan
        )

        row = {
            "n_months": len(actual),
            "n_games": group["game_name"].nunique(),
            "R2": r2,
            "RMSE": np.sqrt(np.mean(error ** 2)),
            "MAE": np.mean(absolute_error),
            "Bias_pred_minus_actual": np.mean(error),
            "MAPE_pct": mape,
            "MAPE_nonzero_months": int(nonzero_actual.sum()),
            "WAPE_pct": wape,
            "sMAPE_pct": smape,
        }

        if group_columns:
            if not isinstance(group_key, tuple):
                group_key = (group_key,)
            row.update(dict(zip(group_columns, group_key)))

        rows.append(row)

    return pd.DataFrame(rows)


performance_overall_df = (
    calculate_metrics(perf_evaluation_df, ["model"])
    .sort_values("model")
    .reset_index(drop=True)
)

performance_by_game_df = (
    calculate_metrics(perf_evaluation_df, ["model", "game_name"])
    .sort_values(["game_name", "model"])
    .reset_index(drop=True)
)

display(performance_overall_df)
display(performance_by_game_df)